In [21]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
df = pd.read_csv("../Data/Raw/IMDB-Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [23]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [24]:
df['review'].str.split().str.len().describe()

count    50000.000000
mean       231.156940
std        171.343997
min          4.000000
25%        126.000000
50%        173.000000
75%        280.000000
max       2470.000000
Name: review, dtype: float64

In [25]:
print(df[df["sentiment"] == "positive"]["review"].iloc[0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [26]:
print(df[df["sentiment"] == "negative"]["review"].iloc[0])

Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.


In [27]:
import re

def remove_text(text):
  text = text.lower()
  text = re.sub(r"<.*?>", "",text)
  text = re.sub(r"[^a-z\s]", "", text)
  text = text.strip()
  return text

In [28]:
sample = df["review"].iloc[0]
print("BEFORE:", sample[:200])
print("AFTER", remove_text(sample)[:200])

BEFORE: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo
AFTER one of the other reviewers has mentioned that after watching just  oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its bru


In [29]:
from collections import Counter

# Clean the reviews
df["cleaned"] = df["review"].apply(remove_text)

# Count every word accross all reviews
all_words = []
for review in df["cleaned"]:
  all_words.extend(review.split())

counter = Counter(all_words)
print(f"Most common words: {counter.most_common(10)}")
print(f"Number of unique words: {len(counter)}")

Most common words: [('the', 650812), ('and', 319428), ('a', 319263), ('of', 288081), ('to', 266297), ('is', 210068), ('in', 183153), ('it', 151365), ('i', 145576), ('this', 145500)]
Number of unique words: 214620


In [30]:
MIN_WORD_FREQ = 2
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in counter.items():
  if count >= MIN_WORD_FREQ:
    vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 81589


In [31]:
# Encode the reviews
MAX_LEN = 400
def encode(text, vocab, max_len=MAX_LEN):
  tokens = text.split()[:max_len]
  ids = [vocab.get(t,1) for t in tokens]
  ids += [0] * (max_len - len(ids))
  return ids


In [32]:
# Test the encoding on a sample review
sample_cleaned = remove_text(df["review"].iloc[0])
encoded = encode(sample_cleaned, vocab)

print("CLEANED:", sample_cleaned[:200])
print("ENCODED:", encoded[:20])

CLEANED: one of the other reviewers has mentioned that after watching just  oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its bru
ENCODED: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]


In [33]:
# Convert labels to numbers
df["label"] = df["sentiment"].apply(lambda x: 1 if x == "positive" else 0)
print(df[["sentiment", "label"]].head())

  sentiment  label
0  positive      1
1  positive      1
2  positive      1
3  negative      0
4  positive      1


In [34]:
%pip install torch

import sys
sys.path.append("../model")

from dataset import Vocabulary, SentimentDataset

# Build vocabulary from your dataframe
vocab = Vocabulary(min_freq=2)
vocab.build(df["review"].tolist())

# Test encoding one review
sample = df["review"].iloc[0]
encoded = vocab.encode(sample)
print("Encoded length:", len(encoded))       # must be 200
print("First 10 ids:", encoded[:10])
print("Vocab size:", len(vocab))

# Test the dataset class
dataset = SentimentDataset(df, vocab)
print("Dataset size:", len(dataset))

# Get one item — this is what PyTorch will call during training
ids, label = dataset[0]
print("Tensor shape:", ids.shape)   # should be torch.Size([200])
print("Label:", label)              # should be 0 or 1

Note: you may need to restart the kernel to use updated packages.
Vocabulary size: 81589
Encoded length: 200
First 10 ids: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Vocab size: 81589
Dataset size: 50000
Tensor shape: torch.Size([200])
Label: tensor(1)


In [35]:
import os
os.makedirs("../saved", exist_ok=True)
vocab.save("../saved/vocab.pkl")

loaded_vocab = Vocabulary.load("../saved/vocab.pkl")
print("Loaded vocab size:", len(loaded_vocab))

print(vocab.encode(sample)[:10])
print(encode(sample, loaded_vocab)[:10])

Vocabulary saved to ../saved/vocab.pkl
Loaded vocab size: 81589
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[1, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [36]:
from network import SentimentModel

model = SentimentModel(vocab_size=len(vocab))

print(model)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total:,}")

SentimentModel(
  (embedding): Embedding(81589, 128)
  (lstm): LSTM(128, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=2, bias=True)
)

Total trainable parameters: 11,234,946


In [38]:
import torch
fake_batch = torch.randint(0, len(vocab), (4, 200))

output = model(fake_batch)

print("Input shape:", fake_batch.shape)
print("Output shape:", output.shape)       
print("Output values:", output)

Input shape: torch.Size([4, 200])
Output shape: torch.Size([4, 2])
Output values: tensor([[-5.8882e-05, -7.9871e-02],
        [-3.2245e-02,  1.2241e-01],
        [-1.5328e-02, -1.4003e-02],
        [ 1.2363e-02,  4.1731e-02]], grad_fn=<AddmmBackward0>)
